# Phase 4 — MongoDB

Loads the results computed in phases 3a and 3b into MongoDB, together with a
sample of the raw reviews, and queries them.

Spark writes to MongoDB directly through the official connector, so the data
never has to pass through the driver — the same approach would hold if the
collections were far larger than they are here.

Requires HDFS and `mongod` to be running (see `docs/setup.md`). The first run
downloads the connector JAR, so it needs an internet connection.

In [1]:
import getpass

from pyspark.sql import SparkSession

In [2]:
USER = getpass.getuser()
HDFS_BASE = f"hdfs://localhost:9000/user/{USER}/steam"
ANALYSIS_PATH = f"{HDFS_BASE}/output/analysis"
INPUT_PATH = f"{HDFS_BASE}/streaming_input/*.csv"

MONGO_URI = "mongodb://127.0.0.1:27017"
MONGO_DB = "steam_reviews"

spark = (
    SparkSession.builder
    .appName("steam-reviews-mongodb")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark

26/09/09 17:01:11 WARN Utils: Your hostname, luca-Katana-15-B13VFK resolves to a loopback address: 127.0.1.1; using 192.168.1.18 instead (on interface wlo1)
26/09/09 17:01:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/luca/.ivy2/cache
The jars for the packages stored in: /home/luca/.ivy2/jars
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-28616a4e-e072-48a1-9dd2-3983e0be9186;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.12;10.3.0 in central
	found org.mongodb#mongodb-driver-sync;4.8.2 in central
	[4.8.2] org.mongodb#mongodb-driver-sync;[4.8.1,4.8.99)
	found org.mongodb#bson;4.8.2 in central
	found org.mongodb#mongodb-driver-core;4.8.2 in central
	found org.mongodb#bson-record-codec;4.8.2 in central
:: resolution report :: resolve 2695ms :: artifacts dl 12ms
	:: modules in use:
	org.mongodb#bson;4.8.2 from central in [default]
	org.mongodb#bson-record-codec;4.8.2 from central in [default]
	org.mongodb#mongodb-driver-core;4.8.2 from central in [default]
	org.mongodb#mongodb-driver-sync;4.8.2 from central in [default]
	org.mongodb.spark#mongo-spark-connector_2.1

## Loading the aggregates

The six Parquet datasets written in phases 3a and 3b become six collections.
`overwrite` keeps the notebook re-runnable without piling up duplicates.

In [3]:
def write_to_mongo(df, collection):
    (
        df.write
        .format("mongodb")
        .mode("overwrite")
        .option("spark.mongodb.write.connection.uri", MONGO_URI)
        .option("database", MONGO_DB)
        .option("collection", collection)
        .save()
    )
    print(f"written: {MONGO_DB}.{collection}  ({df.count()} docs)")


AGGREGATES = [
    "recommendation_by_playtime",
    "recommendation_by_playtime_price",
    "hours_by_outcome",
    "recommendation_by_year",
    "model_metrics",
    "feature_importances",
]

for name in AGGREGATES:
    write_to_mongo(spark.read.parquet(f"{ANALYSIS_PATH}/{name}"), name)

written: steam_reviews.recommendation_by_playtime  (5 docs)
written: steam_reviews.recommendation_by_playtime_price  (20 docs)
written: steam_reviews.hours_by_outcome  (8 docs)
written: steam_reviews.recommendation_by_year  (10 docs)
written: steam_reviews.model_metrics  (2 docs)
written: steam_reviews.feature_importances  (7 docs)


## Loading a sample of the raw reviews

The aggregates alone would only allow trivial queries, so a slice of the
review-level data goes in as well.

In [4]:
from pyspark.sql import functions as F

raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(INPUT_PATH)
)

reviews_sample = (
    raw
    .select(
        "app_id", "title", "hours", "is_recommended", "helpful",
        "price_final", "price_bucket", "positive_ratio", "date",
    )
    .sample(fraction=0.1, seed=42)
)

write_to_mongo(reviews_sample, "reviews")

written: steam_reviews.reviews  (50097 docs)


## Reading back

A round-trip check that the connector wrote what we expect.

In [5]:
check = (
    spark.read
    .format("mongodb")
    .option("spark.mongodb.read.connection.uri", MONGO_URI)
    .option("database", MONGO_DB)
    .option("collection", "recommendation_by_playtime")
    .load()
)

check.show()

+--------------------+---------------+------+-----------+------+
|                 _id|playtime_bucket|  rate|recommended| total|
+--------------------+---------------+------+-----------+------+
|6aa174caae88483f3...|           0-1h| 0.351|       2326|  6626|
|6aa174caae88483f3...|        20-100h|0.8661|     137418|158660|
|6aa174caae88483f3...|           1-5h|0.5748|      10625| 18484|
|6aa174caae88483f3...|          100h+|0.8711|     218341|250650|
|6aa174caae88483f3...|          5-20h|0.8221|      53913| 65580|
+--------------------+---------------+------+-----------+------+



## Querying with pymongo

Spark handled the writing; the queries below go straight through the Mongo
driver, which is how an application consuming these results would do it.

In [6]:
import pandas as pd
from pymongo import MongoClient

client = MongoClient(MONGO_URI)
db = client[MONGO_DB]

print("collections:", db.list_collection_names())
print("reviews:", db.reviews.count_documents({}), "documents")

collections: ['recommendation_by_playtime_price', 'reviews', 'recommendation_by_year', 'feature_importances', 'hours_by_outcome', 'recommendation_by_playtime', 'model_metrics']
reviews: 50097 documents


### Query 1 — invested but unconvinced

Reviews from players with more than 100 hours who still did not recommend
the game, ranked by how many people found the review helpful. These are the
cases the recommendation-rate figures average away.

In [7]:
query = {"hours": {"$gt": 100}, "is_recommended": False, "helpful": {"$gt": 0}}

docs = (
    db.reviews
    .find(query, {"_id": 0, "title": 1, "hours": 1, "helpful": 1, "positive_ratio": 1})
    .sort("helpful", -1)
    .limit(10)
)

pd.DataFrame(list(docs))

,title,hours,helpful,positive_ratio
0,PUBG: BATTLEGROUNDS,348.8,5126,57
1,Grand Theft Auto V,229.2,3521,86
2,Mount & Blade II: Bannerlord,150.6,2859,87
3,DayZ,447.4,1737,74
4,PUBG: BATTLEGROUNDS,219.5,1729,57
5,Team Fortress 2,284.0,1151,93
6,Counter-Strike: Global Offensive,844.7,916,88
7,War Thunder,439.9,741,75
8,The Elder Scrolls V: Skyrim Special Edition,853.7,693,94
9,The Sims™ 3,303.7,663,86


### Query 2 — recommendation rate by price bucket

The aggregation-framework version of what Spark computed: group the raw
reviews, average the boolean, and sort.

In [8]:
pipeline = [
    {
        "$group": {
            "_id": "$price_bucket",
            "reviews": {"$sum": 1},
            "rate": {"$avg": {"$cond": ["$is_recommended", 1, 0]}},
            "avg_hours": {"$avg": "$hours"},
        }
    },
    {"$sort": {"rate": -1}},
]

rows = [
    {
        "price_bucket": r["_id"],
        "reviews": r["reviews"],
        "rate": round(r["rate"], 4),
        "avg_hours": round(r["avg_hours"], 1),
    }
    for r in db.reviews.aggregate(pipeline)
]

pd.DataFrame(rows)

,price_bucket,reviews,rate,avg_hours
0,low,2400,0.9242,132.9
1,mid,16819,0.8879,212.6
2,high,20932,0.8567,176.9
3,free,9946,0.7316,248.6


### Query 3 — recommendation rate by year

`$year` extracts the year from the date field, which the connector stored as
a proper date type rather than a string.

In [9]:
pipeline = [
    {"$match": {"date": {"$ne": None}}},
    {
        "$project": {
            "year": {"$year": "$date"},
            "recommended": {"$cond": ["$is_recommended", 1, 0]},
            "hours": 1,
        }
    },
    {
        "$group": {
            "_id": "$year",
            "reviews": {"$sum": 1},
            "rate": {"$avg": "$recommended"},
            "avg_hours": {"$avg": "$hours"},
        }
    },
    # drop years too thin to read anything into
    {"$match": {"reviews": {"$gte": 50}}},
    {"$sort": {"_id": 1}},
]

rows = [
    {
        "year": r["_id"],
        "reviews": r["reviews"],
        "rate": round(r["rate"], 4),
        "avg_hours": round(r["avg_hours"], 1),
    }
    for r in db.reviews.aggregate(pipeline)
]

pd.DataFrame(rows)

,year,reviews,rate,avg_hours
0,2013,248,0.9435,284.1
1,2014,981,0.9297,260.9
2,2015,1314,0.8090,288.4
3,2016,2274,0.7731,255.5
4,2017,2683,0.7238,277.6
5,2018,2431,0.7729,270.1
6,2019,3981,0.8794,261.1
7,2020,9139,0.8793,213.7
8,2021,10392,0.8830,193.1
9,2022,16617,0.8318,142.8


In [10]:
client.close()
spark.stop()